In [8]:
import cv2
import os
import re
from collections import defaultdict
from tqdm import tqdm

def parse_hit_frames_from_string(input_str):
    """Parses a string of the form 'label: frame, frame, ...' into a dict[label] = list[frames]"""
    label_dict = defaultdict(list)
    lines = input_str.strip().splitlines()

    for line in lines:
        match = re.match(r"(\w+):\s*(.*)", line)
        if match:
            label = match.group(1).lower()
            frames = match.group(2)
            frame_list = [int(f.strip()) for f in frames.split(',') if f.strip()]
            label_dict[label].extend(frame_list)

    return label_dict

def extract_labeled_clips_from_dict(
    video_path,
    label_dict,
    output_dir,
    pre_sec=0.5,
    num_frames=36
):
    os.makedirs(output_dir, exist_ok=True)
    video_id = os.path.splitext(os.path.basename(video_path))[0].lower()

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Failed to open video: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    for label in tqdm(label_dict, desc="Processing labels"):
        hit_frames = label_dict[label]
        for hit_frame in tqdm(hit_frames, leave=False, desc=f"{label} clips"):
            start_frame = max(0, int(hit_frame - pre_sec * fps))
            end_frame = min(start_frame + num_frames - 1, total_frames - 1)

            out_name = f"{label}_{video_id}_{hit_frame:05d}.mp4"
            out_path = os.path.join(output_dir, out_name)
            out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
            for _ in range(start_frame, end_frame + 1):
                ret, frame = cap.read()
                if not ret:
                    break
                out.write(frame)

            out.release()

    cap.release()
    print("✅ All clips saved.")


In [9]:

label_string = """
serve: 975, 1825, 2125, 2710, 3602, 3918, 4359, 4724, 5541, 6379, 8044, 8440, 10111, 10625, 11750, 12149, 12982, 13418
clear: 2796, 4804, 10198
lift: 617, 3998, 5938, 6012, 6504, 6568, 7096, 8828, 9207, 10271, 11350, 12683, 13555, 13631
smash: 1494, 3307, 5275, 8177, 11457, 12237, 13709
drop: 1435, 2212, 4887, 5629, 6457, 6988, 7586, 11839, 13502
net: 1361, 1536, 2870, 6909, 7210, 8126, 9508, 10338, 12289, 12349
drive: 7155, 13043
"""

video_path = str(os.path.expanduser("~/badminton/scratch/IMG_3212.MP4"))
output_dir = str(os.path.expanduser(f"~/badminton/scratch/3212_clips"))

label_dict = parse_hit_frames_from_string(label_string)
extract_labeled_clips_from_dict(video_path, label_dict, output_dir)

Processing labels: 100%|██████████████████████████████████████████████████████| 7/7 [00:11<00:00,  1.69s/it]

✅ All clips saved.


In [ ]:
# 3214
serve:
clear:
lift:
drop:
smash:

In [ ]:

drop: 414, 549, 871, 1700, 4169, 4304
lift: 1956, 4087,4230, 4633, 5165
clear: 2043, 2449, 2696, 3950, 4772, 5240, 5453
backhand: 4169
forehand drop

